In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Επιλογή dataset
csv_path = Path("data/cleaned_ids2018_sampled.csv")
assert csv_path.exists(), "Dataset not found!"

# Αρχικοποίηση μετρητών
total_rows = 0
missing_values = 0
infinite_values = 0
label_counts = pd.Series(dtype="int64")

# Ανάγνωση σε τμήματα για εξοικονόμηση RAM
for chunk in pd.read_csv(csv_path, chunksize=20000):
    chunk.columns = chunk.columns.str.strip()

    total_rows += len(chunk)

    # Κατανομή Labels
    label_counts = label_counts.add(
        chunk["Label"].value_counts(), fill_value=0
    )

    # Έλεγχος NaN και Infinity
    missing_values += int(chunk.isna().sum().sum())
    numeric = chunk.select_dtypes(include="number")
    infinite_values += int(np.isinf(numeric.to_numpy()).sum())

# Αποτελέσματα
print("=== DATASET INFORMATION ===")
print("Total rows:", total_rows)
print("Total columns:", len(chunk.columns))
print("Missing values:", missing_values)
print("Infinite values:", infinite_values)

print("\n=== LABEL DISTRIBUTION ===")
print(label_counts.sort_index().astype(int))

print("\n=== LABEL PERCENTAGES ===")
print((label_counts / total_rows * 100).sort_index().round(2))


In [ ]:

import joblib
import gc
from sklearn.model_selection import train_test_split

folder = Path("processed")
folder.mkdir(exist_ok=True)
source_path = csv_path

# Χρονικά χαρακτηριστικά που δεν επιτρέπεται να είναι αρνητικά
time_columns = [
    "Flow Duration",
    "Flow IAT Min", "Flow IAT Max",
    "Fwd IAT Min", "Fwd IAT Max",
    "Bwd IAT Min", "Bwd IAT Max"
]

# Έλεγχος ότι Min <= Max
min_max_pairs = [
    ("Flow IAT Min", "Flow IAT Max"),
    ("Fwd IAT Min", "Fwd IAT Max"),
    ("Bwd IAT Min", "Bwd IAT Max"),
    ("Idle Min", "Idle Max")
]

def prepare_chunk(chunk):
    chunk.columns = chunk.columns.str.strip()

    invalid = (chunk[time_columns] < 0).any(axis=1)

    for min_col, max_col in min_max_pairs:
        invalid |= chunk[min_col] > chunk[max_col]

    valid = chunk.loc[~invalid].copy()

    features = valid.drop(columns=["Unnamed: 0", "Label"])
    signatures = pd.util.hash_pandas_object(
        features, index=False
    ).to_numpy()

    return valid, signatures, int(invalid.sum())


# Αναγνώριση μοναδικών ομάδων εγγραφών
labels_by_group = {}
invalid_rows = 0
valid_rows = 0

for chunk in pd.read_csv(source_path, chunksize=20000):
    valid, signatures, invalid_count = prepare_chunk(chunk)

    invalid_rows += invalid_count
    valid_rows += len(valid)

    for signature, label in zip(signatures, valid["Label"]):
        labels_by_group.setdefault(int(signature), int(label))


# Διαχωρισμός 80% Training - 20% Testing
train_groups, test_groups = train_test_split(
    list(labels_by_group),
    test_size=0.20,
    random_state=42,
    stratify=list(labels_by_group.values())
)

train_groups = set(train_groups)
test_groups = set(test_groups)

assert train_groups.isdisjoint(test_groups)

# Αποθήκευση διαχωρισμού
joblib.dump(
    {"train": train_groups, "test": test_groups},
    folder / "group_split.joblib"
)

print("=== GROUP-AWARE SPLIT ===")
print("Invalid rows:", invalid_rows)
print("Valid rows:", valid_rows)
print("Unique groups:", len(labels_by_group))
print("Duplicate feature rows:", valid_rows - len(labels_by_group))
print("Training groups:", len(train_groups))
print("Testing groups:", len(test_groups))
print("Overlapping groups:", len(train_groups & test_groups))

del labels_by_group, train_groups, test_groups
gc.collect()


In [ ]:

# Φόρτωση του διαχωρισμού από το προηγούμενο βήμα
split = joblib.load(folder / "group_split.joblib")
train_groups = split["train"]
test_groups = split["test"]

columns = pd.read_csv(csv_path, nrows=0).columns.str.strip()
columns = columns.drop("Unnamed: 0")

# 1. Εντοπισμός ομάδων με αντικρουόμενα Labels
first_labels = {}
conflicts = set()

for chunk in pd.read_csv(csv_path, chunksize=20000):
    valid, signatures, _ = prepare_chunk(chunk)

    for signature, label in zip(signatures, valid["Label"]):
        h, label = int(signature), int(label)

        if h not in first_labels:
            first_labels[h] = label
        elif first_labels[h] != label:
            conflicts.add(h)

print("Conflicting groups:", len(conflicts))
del first_labels

# 2. Αφαίρεση διπλοτύπων και αποθήκευση Train/Test
train_path = folder / "train_grouped.csv"
test_path = folder / "test_grouped.csv"

for path in [train_path, test_path]:
    path.unlink(missing_ok=True)

seen = set()
counts = {"train": 0, "test": 0}

for chunk in pd.read_csv(csv_path, chunksize=20000):
    valid, signatures, _ = prepare_chunk(chunk)
    destinations = []

    for signature in signatures:
        h = int(signature)

        if h in conflicts or h in seen:
            destinations.append("skip")
            continue

        seen.add(h)

        if h in train_groups:
            destinations.append("train")
        elif h in test_groups:
            destinations.append("test")
        else:
            raise ValueError("Unknown group")

    destinations = np.array(destinations)

    for name, path in [
        ("train", train_path),
        ("test", test_path)
    ]:
        subset = valid.loc[destinations == name, columns]

        if not subset.empty:
            subset.to_csv(
                path,
                mode="a",
                header=not path.exists(),
                index=False
            )
            counts[name] += len(subset)

# 3. Τελικός έλεγχος
expected = len(train_groups) + len(test_groups) - len(conflicts)

assert sum(counts.values()) == expected

print("\n=== CLEANED DATASET ===")
print("Training rows:", counts["train"])
print("Testing rows:", counts["test"])
print("Total rows:", sum(counts.values()))
print("Expected rows:", expected)

del seen, train_groups, test_groups


In [ ]:

import json

threshold = 0.95

# 1. Διαβάζουμε μόνο τα χαρακτηριστικά του Training Set
X = pd.read_csv(
    folder / "train_grouped.csv",
    usecols=lambda col: col != "Label"
)

# 2. Αφαιρούμε χαρακτηριστικά με σταθερή τιμή
constant_features = X.columns[
    X.min() == X.max()
].tolist()

active_features = [
    col for col in X.columns
    if col not in constant_features
]

print("Initial features:", X.shape[1])
print("Constant features removed:", len(constant_features))

# 3. Υπολογίζουμε Pearson Correlation
corr = X[active_features].corr()

del X
gc.collect()

# Κρατάμε μόνο το επάνω τρίγωνο του πίνακα
upper = corr.abs().where(
    np.triu(np.ones(corr.shape, dtype=bool), k=1)
)

# 4. Αφαιρούμε τα ιδιαίτερα συσχετισμένα features
removed_features = upper.columns[
    (upper > threshold).any()
].tolist()

selected_features = [
    col for col in active_features
    if col not in removed_features
]

# 5. Έλεγχος ότι δεν απέμειναν υψηλές συσχετίσεις
selected_corr = corr.loc[
    selected_features, selected_features
].abs()

selected_upper = selected_corr.where(
    np.triu(
        np.ones(selected_corr.shape, dtype=bool),
        k=1
    )
)

remaining_pairs = int(
    (selected_upper > threshold).sum().sum()
)

assert remaining_pairs == 0

print("High-correlation pairs:",
      int((upper > threshold).sum().sum()))
print("Correlated features removed:", len(removed_features))
print("Final selected features:", len(selected_features))
print("Remaining high-correlation pairs:", remaining_pairs)

# 6. Αποθήκευση των χαρακτηριστικών
with open(
    folder / "feature_selection_final.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump({
        "method": "Pearson correlation",
        "threshold": threshold,
        "constant_features": constant_features,
        "removed_features": removed_features,
        "selected_features": selected_features
    }, f, indent=2)

# 7. Δημιουργία τελικών Train/Test με 44 features
columns_to_keep = selected_features + ["Label"]

for split in ["train", "test"]:
    input_path = folder / f"{split}_grouped.csv"
    output_path = folder / f"{split}_final_selected.csv"

    total = 0

    for i, chunk in enumerate(
        pd.read_csv(
            input_path,
            usecols=columns_to_keep,
            chunksize=20000
        )
    ):
        chunk[columns_to_keep].to_csv(
            output_path,
            mode="w" if i == 0 else "a",
            header=(i == 0),
            index=False
        )
        total += len(chunk)

    print(f"{split.upper()} selected rows: {total}")


In [ ]:

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
chunk_size = 20000

# 1. Εκπαίδευση του scaler μόνο στο Training Set
for chunk in pd.read_csv(
    folder / "train_final_selected.csv",
    chunksize=chunk_size
):
    scaler.partial_fit(chunk[selected_features])

# Αποθήκευση scaler και χαρακτηριστικών
joblib.dump(
    {"scaler": scaler, "features": selected_features},
    folder / "minmax_scaler_final.joblib"
)

# 2. Εφαρμογή σε Training και Testing Set
for split in ["train", "test"]:
    input_path = folder / f"{split}_final_selected.csv"
    output_path = folder / f"{split}_final_scaled.csv"

    total_rows = 0
    outside_range = 0

    for i, chunk in enumerate(
        pd.read_csv(input_path, chunksize=chunk_size)
    ):
        values = scaler.transform(
            chunk[selected_features]
        ).astype(np.float32)

        assert np.isfinite(values).all()

        outside_range += int(
            ((values < -1e-5) | (values > 1 + 1e-5)).sum()
        )

        scaled = pd.DataFrame(
            values, columns=selected_features
        )
        scaled["Label"] = chunk["Label"].to_numpy()

        scaled.to_csv(
            output_path,
            mode="w" if i == 0 else "a",
            header=(i == 0),
            index=False
        )

        total_rows += len(scaled)

    print(
        f"{split.upper()}: {total_rows} rows, "
        f"{len(selected_features)} features, "
        f"{outside_range} out-of-range values"
    )

    if split == "train":
        assert outside_range == 0

print("\nMin-Max Scaling completed!")


In [ ]:

import json

# Φόρτωση επιλεγμένων χαρακτηριστικών
with open(folder / "feature_selection_final.json") as f:
    features = json.load(f)["selected_features"]

# Τελικός έλεγχος των δύο datasets
for split, expected_rows in [("train", 715505), ("test", 178877)]:

    selected = pd.read_csv(
        folder / f"{split}_final_selected.csv",
        usecols=["Label"]
    )

    scaled = pd.read_csv(
        folder / f"{split}_final_scaled.csv",
        dtype={col: "float32" for col in features}
    )

    assert len(scaled) == expected_rows
    assert len(features) == 44
    assert list(scaled.columns) == features + ["Label"]

    # Έλεγχος ότι τα Labels παρέμειναν ίδια
    assert np.array_equal(
        selected["Label"].to_numpy(),
        scaled["Label"].to_numpy()
    )

    # Έλεγχος για NaN και Infinity
    assert np.isfinite(scaled[features].to_numpy()).all()

    print(
        f"{split.upper()}: {len(scaled)} rows, "
        f"{len(features)} features - OK"
    )

    del selected, scaled

print("\nPreprocessing completed successfully!")
